# Text Embedding with EmbeddingGemma 2 and OpenVINO

[EmbeddingGemma 2](https://huggingface.co/google/embeddinggemma-2) is an open multimodal embedding model from Google DeepMind. It maps text, code, images, video, and audio into a single 768-dimensional vector space. The full model has 740M parameters: a 270M text backbone plus modular vision and audio encoders.

This notebook covers the **text embedding path**: exporting the Sentence Transformers checkpoint to OpenVINO IR with Optimum Intel and running inference with `OVSentenceTransformer`. It also demonstrates the task instruction prefixes and Matryoshka Representation Learning (MRL) truncation that EmbeddingGemma 2 expects.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [About EmbeddingGemma 2](#About-EmbeddingGemma-2)
- [Convert model using Optimum Intel](#Convert-model-using-Optimum-Intel)
- [Select inference device](#Select-inference-device)
- [Prepare static shapes for NPU](#Prepare-static-shapes-for-NPU)
- [Run OpenVINO model inference](#Run-OpenVINO-model-inference)
- [Matryoshka dimension truncation](#Matryoshka-dimension-truncation)
- [Compare with original PyTorch model (optional)](#Compare-with-original-PyTorch-model-optional)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/embeddinggemma-2/embeddinggemma-2.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)


In [ ]:
%pip install -q --no-deps "git+https://github.com/huggingface/optimum-intel.git"
%pip install -q "optimum~=2.3.0" "openvino>=2026.4.1" "openvino-tokenizers>=2026.4.1" "nncf>=3.3.0" "huggingface-hub>=0.23.2,<2.0"
# transformers>=5.19.0 is required for the embedding_gemma2 architecture.
# optimum-intel is installed with --no-deps because its transformers pin is older than the architecture requires.
%pip install -q "transformers==5.19.0" "sentence-transformers==6.1.0" "torch==2.14.1" "Pillow" "requests" --extra-index-url https://download.pytorch.org/whl/cpu

In [ ]:
from pathlib import Path

import requests

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("embeddinggemma-2.ipynb")

## About EmbeddingGemma 2
[back to top ⬆️](#Table-of-contents:)

EmbeddingGemma 2 is trained with short task instruction prefixes for text inputs. Using the expected prefix improves embedding quality. For asymmetric retrieval, queries and documents use different prefixes:

| Use case | Query prefix | Document prefix |
|---|---|---|
| Web / document search | `task: search result \| query: ` | `title: none \| text: ` |
| Question answering | `task: question answering \| query: ` | `title: none \| text: ` |
| Code retrieval | `task: code retrieval \| query: ` | `title: none \| text: ` |

The model also supports Matryoshka Representation Learning. The native 768-dimensional vector can be truncated to 512, 256, or 128 dimensions and re-normalized. Quality is close to lossless down to 256 dimensions.

> **Note:** this notebook exports the Sentence Transformers text pipeline. The modular vision and audio encoders are not covered here.


In [ ]:
MODEL_ID = "google/embeddinggemma-2"
MODEL_REVISION = "914f7f89142e33e77833254d9c9b90c3cef7303b"

QUERY_PREFIX = "task: search result | query: "
DOCUMENT_PREFIX = "title: none | text: "

## Convert model using Optimum Intel
[back to top ⬆️](#Table-of-contents:)

The model is exported with Optimum Intel using the `sentence_transformers` library and the `feature-extraction` task:

```bash
optimum-cli export openvino --model google/embeddinggemma-2 --library sentence_transformers --task feature-extraction --weight-format fp32 --model-kwargs '{"revision": "914f7f89142e33e77833254d9c9b90c3cef7303b"}' <output_dir>
```

`--model-kwargs` pins the encoder checkpoint revision because the OpenVINO exporter does not expose a top-level `--revision` argument for `sentence_transformers`.

The export uses FP32 weights. EmbeddingGemma 2 should not be exported or run in FP16: its activation range exceeds FP16, which can return NaN or silently degraded embeddings instead of raising an error.


In [ ]:
weight_format = "FP32"

In [ ]:
import json
from cmd_helper import optimum_cli
from pathlib import Path

model_export_dir = Path(MODEL_ID.split("/")[-1]) / weight_format

if not model_export_dir.exists():
    optimum_cli(
        MODEL_ID,
        model_export_dir,
        additional_args={
            "library": "sentence_transformers",
            "task": "feature-extraction",
            "weight-format": weight_format.lower(),
            "model-kwargs": json.dumps({"revision": MODEL_REVISION}),
        },
    )

Select the device for OpenVINO inference. `NPU` is available when the OpenVINO NPU plugin and driver are installed. If NPU compilation fails on your driver, fall back to `CPU` or `GPU`. NPU compilation also requires static input shapes; the next cell reshapes the exported IR when NPU is selected.


In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=[])

device

### Prepare static shapes for NPU
[back to top ⬆️](#Table-of-contents:)

NPU compilation in OpenVINO requires static input shapes. The exported IR uses dynamic batch and sequence axes by default. When `NPU` is selected, reshape the text encoder inputs to a fixed sequence length before loading `OVSentenceTransformer`. The same length is used for `model.max_seq_len` below.


In [ ]:
import openvino as ov

NPU_SEQUENCE_LENGTH = 512

if device.value == "NPU":
    core = ov.Core()
    ir_path = model_export_dir / "openvino_model.xml"
    ov_model = core.read_model(ir_path)
    static_inputs = {}
    for port in ov_model.inputs:
        name = port.get_any_name()
        if name in ("input_ids", "attention_mask", "token_type_ids"):
            static_inputs[name] = [1, NPU_SEQUENCE_LENGTH]
    if not static_inputs:
        raise RuntimeError("Could not find text encoder inputs to reshape for NPU.")
    ov_model.reshape(static_inputs)
    ov.save_model(ov_model, ir_path)
    print(f"Reshaped OpenVINO IR for NPU to sequence length {NPU_SEQUENCE_LENGTH}.")
else:
    print("Static reshaping is only required for NPU; skipping.")

## Run OpenVINO model inference
[back to top ⬆️](#Table-of-contents:)

Load the exported model with `OVSentenceTransformer` and encode a small query/document set. The embeddings are L2-normalized, so a dot product equals cosine similarity.


In [ ]:
from optimum.intel import OVSentenceTransformer

model = OVSentenceTransformer.from_pretrained(model_export_dir, device=device.value)
model.max_seq_len = 512

In [ ]:
import numpy as np

QUERY = "What causes the northern lights?"
DOCUMENTS = [
    "Charged particles from the sun excite gases in the upper atmosphere.",
    "Mars is often called the Red Planet because of iron oxide on its surface.",
    "The capital of China is Beijing.",
]


def l2_normalize(embeddings: np.ndarray) -> np.ndarray:
    """L2-normalize embeddings along the last axis."""
    return embeddings / np.linalg.norm(embeddings, axis=-1, keepdims=True)


def rank_documents(model: OVSentenceTransformer, query: str, documents: list[str]) -> list[tuple[str, float]]:
    """Encode a query and documents with task prefixes and return ranked scores."""
    query_embedding = model.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0]
    document_embeddings = model.encode([DOCUMENT_PREFIX + document for document in documents], normalize_embeddings=True)
    scores = document_embeddings @ query_embedding
    return sorted(zip(documents, scores.tolist()), key=lambda item: item[1], reverse=True)


for document, score in rank_documents(model, QUERY, DOCUMENTS):
    print(f"{score:.4f}  {document}")

## Matryoshka dimension truncation
[back to top ⬆️](#Table-of-contents:)

EmbeddingGemma 2 supports truncated representations. Slice the leading dimensions, re-normalize, and use the same dimension for queries and documents. The ranking below uses 256 dimensions instead of the native 768.


In [ ]:
TRUNCATION_DIM = 256


def rank_documents_truncated(model: OVSentenceTransformer, query: str, documents: list[str], dim: int) -> list[tuple[str, float]]:
    """Encode inputs, truncate to `dim`, re-normalize, and return ranked scores."""
    query_embedding = l2_normalize(model.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0][:dim])
    document_embeddings = l2_normalize(model.encode([DOCUMENT_PREFIX + document for document in documents], normalize_embeddings=True)[:, :dim])
    scores = document_embeddings @ query_embedding
    return sorted(zip(documents, scores.tolist()), key=lambda item: item[1], reverse=True)


for document, score in rank_documents_truncated(model, QUERY, DOCUMENTS, TRUNCATION_DIM):
    print(f"{score:.4f}  {document}")

## Compare with original PyTorch model (optional)
[back to top ⬆️](#Table-of-contents:)

Enable the checkbox below to load the original PyTorch Sentence Transformer model and compare its embeddings with the OpenVINO result. The comparison reports the minimum row-wise cosine similarity and the maximum absolute difference after L2 normalization.


In [ ]:
import ipywidgets as widgets

run_pt_compare = widgets.Checkbox(value=False, description="Run PyTorch comparison")

run_pt_compare

In [ ]:
if not run_pt_compare.value:
    print("Skipping PyTorch comparison. Tick the checkbox above and re-run this cell to enable.")
else:
    from sentence_transformers import SentenceTransformer

    pt_model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION, device="cpu")
    pt_model.max_seq_len = 512

    texts = [QUERY_PREFIX + QUERY] + [DOCUMENT_PREFIX + document for document in DOCUMENTS]
    pt_embeddings = l2_normalize(pt_model.encode(texts, normalize_embeddings=True))
    ov_embeddings = l2_normalize(model.encode(texts, normalize_embeddings=True))

    min_cosine = float(np.min(np.sum(pt_embeddings * ov_embeddings, axis=1)))
    max_abs_diff = float(np.max(np.abs(pt_embeddings - ov_embeddings)))
    print(f"OpenVINO vs PyTorch: min row cosine={min_cosine:.8f}, max abs={max_abs_diff:.8f}")

## Clean up
[back to top ⬆️](#Table-of-contents:)

Uncomment and run the cell below to remove the exported OpenVINO model and the downloaded Hugging Face files.


In [ ]:
# import shutil
# shutil.rmtree(model_export_dir, ignore_errors=True)